# 02 · Preprocessing and feature engineering
Implemented in `src/data_prep.py` and `src/pipeline.py`.

In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40, 'display.width', 160)
T = '../reports/tables/'; F = '../reports/figures/'
pd.read_csv(T+'data_dictionary_processed.csv')

,column,dtype,missing_pct,n_unique,example
0,Accident_Index,str,0.000,2046815,200501BS00001
1,year,int64,0.000,13,2005
2,date,datetime64[us],0.000,4748,2005-01-04 00:00:00
3,severity,int64,0.000,3,1
4,ksi,int64,0.000,2,1
5,sev_weight,int64,0.000,3,5
6,lat,float32,0.000,814984,51.489098
7,lon,float32,0.000,1417552,-0.19117
8,Number_of_Casualties,int64,0.000,52,1
9,Did_Police_Officer_Attend_Scene_of_Accident,float64,0.014,3,1.0


## Leak-free spatial history
For an accident in year *Y*, every `h_*` feature uses only accidents from years before *Y* (expanding window). Training rows never see their own label.

In [2]:
df = pd.read_parquet('../data/processed/accidents_clean.parquet')
df[[c for c in df.columns if c.startswith('h_')] + ['year', 'severity']].groupby('year').mean().round(3)

,h_cell_n,h_cell_ksi_rate,h_cell_score,h_road_n,h_road_ksi_rate,h_road_score,h_la_n,h_la_ksi_rate,h_la_score,h_dist_hotspot_m,h_in_hotspot,severity
year,,,,,,,,,,,,
2005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.155
2006,4.293000,0.137,6.417000,123.017998,0.138,191.531998,757.164001,0.140,1201.212036,27083.460938,0.028,0.163
2007,8.248000,0.139,12.545000,235.123993,0.142,369.059998,1471.723022,0.144,2359.847900,16835.230469,0.052,0.163
2008,12.380000,0.139,18.968000,339.132996,0.144,534.224976,2162.875977,0.146,3481.033936,12791.407227,0.075,0.163
2009,16.282000,0.140,25.011999,430.654999,0.144,679.833984,2761.152100,0.146,4447.007812,12051.960938,0.089,0.160
2010,20.466999,0.139,31.407000,515.213989,0.144,812.239990,3382.980957,0.146,5445.221191,10168.189453,0.105,0.155
2011,24.801001,0.138,37.952000,586.966980,0.144,927.241028,3965.134033,0.146,6377.437012,9915.959961,0.120,0.162
2012,28.563999,0.138,43.678001,664.276001,0.145,1054.264038,4549.467773,0.147,7341.866211,9724.206055,0.128,0.166
2013,32.361000,0.139,49.646000,751.838989,0.147,1198.864014,5155.548828,0.148,8337.406250,9541.175781,0.135,0.165


In [3]:
from src.pipeline import feature_set, make_preprocessor
cats, nums = feature_set('full')
print(len(cats), 'categorical +', len(nums), 'numeric features')
make_preprocessor(cats, nums)

17 categorical + 26 numeric features


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",1.0
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``